In [267]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.model_selection import KFold


In [ ]:
df_1_1 = pd.read_csv("1_to_1.csv")
df_1_5 = pd.read_csv("1_to_5.csv")


1
1


In [303]:
def evaluate_model(X, y, model, n_runs=10):
    f1_scores = []
    conf_matrices = []

    for i in range(n_runs):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.3,
            random_state=i,
            stratify=y
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # F1
        f1_scores.append(f1_score(y_test, y_pred))

        # Confusion matrix
        conf_matrices.append(confusion_matrix(y_test, y_pred))

    # Convert to numpy array for averaging
    conf_matrices = np.array(conf_matrices)

    return (
        np.mean(f1_scores),
        np.std(f1_scores),
        np.mean(conf_matrices, axis=0)   
    )


In [ ]:

def in_distribution_hashtags(df, n_runs=10):
    print("In-Distribution")
    results = {}
    model = XGBClassifier(
                max_depth=8,
                learning_rate=0.3,
                n_estimators=100,
                scale_pos_weight=3,
                min_child_weight=1,
                subsample=1.0,
                objective="binary:logistic",
                random_state=42,
                eval_metric="logloss",
                tree_method="hist"
             )
    
    for tag in df["hashtag"].unique():

        df_tag = df[df["hashtag"] == tag]


        X = df_tag.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
        y = df_tag["label"]

        mean_f1, std_f1, avg_conf = evaluate_model(X, y, model, n_runs=n_runs)

        results[tag] = (mean_f1, std_f1)

        print(f"Hashtag: {tag}")
        print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
        print("-" * 40)



In [306]:


def out_of_distribution_hashtags(df):

    hashtags = df["hashtag"].unique()
    print("Out-of-Distribution")
    results = {}

    model = XGBClassifier(
                max_depth=8,
                learning_rate=0.3,
                n_estimators=100,
                scale_pos_weight=3,
                min_child_weight=1,
                subsample=1.0,
                objective="binary:logistic",
                random_state=42,
                eval_metric="logloss",
                tree_method="hist"
             )
    
    for tag in hashtags:

        test_df = df[df["hashtag"] == tag].copy()
        train_df = df[df["hashtag"] != tag].copy()
        print(len(train_df), len(test_df))

        X_test = test_df.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
        y_test = test_df["label"]

        f1_scores = []

        kf = KFold(n_splits=10, shuffle=True, random_state=42)

        for train_index, _ in kf.split(train_df):

            train_subset = train_df.iloc[train_index]

            X_train = train_subset.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
            y_train = train_subset["label"]

            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            f1_scores.append(f1_score(y_test, y_pred))

        mean_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores)

        print(f"Hashtag: {tag}")
        print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
        print("-" * 40)

        results[tag] = (mean_f1, std_f1)





In [307]:
out_of_distribution_hashtags(df_1_1)

Out-of-Distribution
62761 7289
Hashtag: aew
F1: 0.6933 ± 0.0019
----------------------------------------
66925 3125
Hashtag: TheTraitors
F1: 0.6995 ± 0.0021
----------------------------------------
62426 7624
Hashtag: AI
F1: 0.7108 ± 0.0030
----------------------------------------
61042 9008
Hashtag: Trump
F1: 0.6989 ± 0.0019
----------------------------------------
55001 15049
Hashtag: booksky
F1: 0.7112 ± 0.0014
----------------------------------------
59757 10293
Hashtag: Minneapolis
F1: 0.7071 ± 0.0023
----------------------------------------
56388 13662
Hashtag: gaza
F1: 0.6921 ± 0.0021
----------------------------------------
69903 147
Hashtag: releasetheepsteinfiles
F1: 0.7281 ± 0.0189
----------------------------------------
66790 3260
Hashtag: GoldenGlobes
F1: 0.7088 ± 0.0040
----------------------------------------
69457 593
Hashtag: tennis
F1: 0.7073 ± 0.0049
----------------------------------------


In [ ]:
in_distribution_hashtags(df_1_1)

Hashtag: aew
F1: 0.6818 ± 0.0058
----------------------------------------
Hashtag: TheTraitors
F1: 0.6548 ± 0.0131
----------------------------------------
Hashtag: AI
F1: 0.7071 ± 0.0047
----------------------------------------
Hashtag: Trump
F1: 0.6764 ± 0.0090
----------------------------------------
Hashtag: booksky
F1: 0.7062 ± 0.0057
----------------------------------------
Hashtag: Minneapolis
F1: 0.6862 ± 0.0056
----------------------------------------
Hashtag: gaza
F1: 0.6963 ± 0.0036
----------------------------------------
Hashtag: releasetheepsteinfiles
F1: 0.6303 ± 0.0740
----------------------------------------
Hashtag: GoldenGlobes
F1: 0.6661 ± 0.0075
----------------------------------------
Hashtag: tennis
F1: 0.6182 ± 0.0378
----------------------------------------


In [304]:
#mixed 
model = XGBClassifier(
    max_depth=8,
    learning_rate=0.3,
    n_estimators=100,
    scale_pos_weight=1,
    min_child_weight=1,
    subsample=1.0,
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss",
    tree_method="hist"
)

id_cols = ["A_id", "S_id", "P_id", "hashtag"]

X = df_1_5.drop(columns=id_cols + ["label"])
y = df_1_5["label"]
X = X.fillna(0)


mean_f1, std_f1, avg_conf = evaluate_model(X, y, model, n_runs=10)
print(f"Mixed 1:5:")
print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
print("-" * 40)


X = df_1_1.drop(columns=id_cols + ["label"])
y = df_1_1["label"]
X = X.fillna(0)


mean_f1, std_f1, avg_conf = evaluate_model(X, y, model, n_runs=10)
print(f"Mixed 1:1:")
print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
print("-" * 40)


Mixed 1:5:
F1: 0.4035 ± 0.0051
----------------------------------------
Mixed 1:1:
F1: 0.6768 ± 0.0029
----------------------------------------


In [298]:

booster = model.get_booster()

importance_dict = booster.get_score(importance_type="gain")

importance_df = pd.DataFrame({
    "feature": importance_dict.keys(),
    "gain": importance_dict.values()
}).sort_values(by="gain", ascending=False)

print(importance_df)


                   feature       gain
21    U-HA_R_RetweetedRate  24.072832
19   U-HA_R_RetweetPercent   9.875425
10        U-P_R_ProfileUrl   7.927974
9        U-P_R_TweetNumDay   7.799597
2      U-P_R_activeBeforeP   6.415478
20  U-HA_R_AverageInterval   5.927243
7     U-P_R_FollowerNumDay   5.669682
8     U-P_R_FolloweeNumDay   5.142353
4        U-P_R_FollowerNum   4.871524
6           U-P_R_TweetNum   4.847560
5        U-P_R_FolloweeNum   4.747746
3         U-P_R_AccountAge   4.588258
0            U-P_R_FollowS   3.387034
16    U-P_S_FolloweeNumDay   3.291378
15    U-P_S_FollowerNumDay   3.220591
24    U-HA_S_RetweetedRate   3.154616
1     U-P_SR_followersDiff   3.135186
17       U-P_S_TweetNumDay   3.124118
23  U-HA_S_AverageInterval   3.092262
12       U-P_S_FollowerNum   3.091574
14          U-P_S_TweetNum   3.076617
22   U-HA_S_RetweetPercent   3.072881
13       U-P_S_FolloweeNum   3.043938
11        U-P_S_AccountAge   3.035679
18        U-P_S_ProfileUrl   2.857439
